# Structure viewer

Interactive 3D view of a PDB biological assembly or an AlphaFold model (`dsa.structure_viewer.viewer.StructureViewer`).

- **Move it:** drag to rotate, scroll to zoom, right-drag (or ctrl-drag) to translate.
- **Hover a residue** to see: bio chain, author chain, protein (UniProt), UniProt residue number, residue name and
  single letter, and SASA (ASA, relative ASA, and NZ for lysines) from
  - the **assembly** (multimer SASA, burial by other chains included),
  - the **chain alone** (the same PDB chain without its partners),
  - the **AlphaFold monomer** at the same UniProt residue.
- Hovering also **highlights the residues in contact**, coloured by bond type: salt bridge, hydrogen bond, cation–π,
  covalent isopeptide, contacts ≤ 4.5 Å / ≤ 6.5 Å (the checked types; with none checked, the four bonds and contacts
  ≤ 4.5 Å). A dashed line joins the closest atom pair.
- **Lysines** highlights lysines (dark sticks with a translucent halo); the rest of the structure keeps its chain
  colours, only lighter. The filters below it combine with AND:
  - **Chains**, grouped by UniProt (the UniProt box switches all its chains): only lysines on the checked chains;
  - **Bond types**: only lysines with any of the checked types (none checked: no filter);
  - **Bond source**: only lysines with such a bond from a checked source, *Inter-chain (PDB)*, *Intra-chain (PDB)* or
    *Intra-chain (AlphaFold)* (none checked: no filter). On a PDB structure the AlphaFold contacts of each protein are
    placed on every chain copy by UniProt number, where both residues match.
  Bonded partners are not drawn until you hover or pin a lysine.
- **Lysines shown** lists the highlighted lysines; clicking one pins it and zooms in without changing the highlighting.
- **AlphaFold panel** (PDB structures): when exactly one chain is checked, a second view on the left shows the AlphaFold
  model of that chain's UniProt id, superposed on the chain (least squares on the CA atoms matched by UniProt number;
  the RMSD is shown) and rotated / zoomed together with it. The same Lysines / bond type / bond source controls apply to
  it, using the AlphaFold model's own contacts; hovering or pinning a residue in one view marks the residue at the same
  UniProt position in the other. *Colour by pLDDT* shows AlphaFold's confidence. `alphafold_panel=False` leaves it out.
- **Pinning** a residue also shows its neighbourhood (protein atoms within 8 Å) as van der Waals spheres (FreeSASA's
  ProtOr radii) and, as green dots, every position where the centre of a 1.4 Å probe touching the lysine NZ (other
  residues: any of their atoms) fits without overlapping another atom; a probe sphere sits at the most open position
  (red if there is none). This is the Shrake–Rupley calculation done live: the panel shows "k of N positions open ≈ X Å²",
  which agrees with the stored FreeSASA NZ values (mean difference < 1 Å² on 4ah6 and 3r45). Uncheck *On pin: van der
  Waals view* to turn it off.
- **Click** a residue to pin it (hovering then stops changing the selection); click again or press *Clear* to release.

Contacts are the saved lysine contacts (a lysine NZ within 6.5 Å of the partner), so bonds appear on lysines and
on their partners. 3Dmol.js is inlined into the page (`vendor/`), so no internet access is needed.

In [ ]:
import warnings
from pathlib import Path
from dsa.structure_viewer.viewer import StructureViewer

# the UniProt aligner warns for structure residues outside the PDB-UniProt alignment (e.g. tags)
warnings.filterwarnings("ignore", message="Invalid residue id")
PAGES = Path("_pages")   # standalone pages saved by viewer.save(); gitignored
PAGES.mkdir(exist_ok=True)

## PDB homodimer: `4ah6` (Q6PI48)

K224 is at the dimer interface: hover it to see its inter-chain partner E625 of the other chain.

In [ ]:
viewer = StructureViewer.from_pdb("4ah6")
viewer.show()

### The same values as tables

`viewer.residues` is exactly what the hover panel shows; `viewer.bonds` lists the residue pairs in contact.

In [ ]:
residues = viewer.residues
residues[(residues.resname3 == "LYS") & (residues.resnum == 224)]

In [ ]:
bonds = viewer.bonds
bonds[bonds.residue1.str.startswith("LYS 224 ") | bonds.residue2.str.startswith("LYS 224 ")][["residue1", "residue2", "types", "dist"]]

## Heteromer: `3r45` (HJURP Q8NCD3 + CENP-A P49450 + histone H4 P62805)

`color_by="protein"` colours each UniProt id instead of each chain.

In [ ]:
StructureViewer.from_pdb("3r45", color_by="protein").show()

## AlphaFold model: Q6PI48

Only the AlphaFold SASA is shown; contacts are the intra-chain contacts of the AlphaFold model.

In [ ]:
StructureViewer.from_alphafold("Q6PI48").show()

## Modified lysines: `2vnf`

Lysines are drawn as sticks in a darker shade of their chain's colour, on a lighter cartoon. That includes modified
lysines, here trimethyl-lysine (M3L) in the peptide chain B1.

In [ ]:
viewer_2vnf = StructureViewer.from_pdb("2vnf")
r = viewer_2vnf.residues
display(r[r.is_lysine][["bio_chain", "resnum", "resname3", "resname1", "uniprot_id", "unp_resnum", "multimer_asa_nz"]])
viewer_2vnf.show()

## Part of an assembly

`chains=[...]` keeps only some bio chains, for large assemblies. The SASA values do not change (they are computed on
the whole assembly or the whole chain); contacts to chains that are not shown are left out.

In [ ]:
StructureViewer.from_pdb("8cn1", chains=["A1", "B1", "C1", "D1"]).show()

## Standalone page

`save` writes one self-contained HTML file (apart from 3Dmol.js), to open in any browser or share.

In [ ]:
viewer.save(PAGES / "4ah6.html")